In [5]:
pip install pymupdf pdf2image pytesseract pillow requests


In [19]:
import fitz  # PyMuPDF for PDF processing
#import pytesseract  # OCR for image-based PDFs
#from pdf2image import convert_from_path  # Convert PDF pages to images
#from PIL import Image, ImageEnhance, ImageFilter
import requests
import re


In [20]:


# Groq API details (Replace with actual API key and endpoint)
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
API_KEY = ""


def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF, using OCR if necessary."""
    doc = fitz.open(pdf_path)
    full_text = ""

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text")

        if text.strip():
            full_text += text + "\n"
        else:
            images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1)
            for img in images:
                img = preprocess_image(img)  # Preprocess image before OCR
                text = pytesseract.image_to_string(img)
                full_text += text + "\n"

    return clean_text(full_text.strip())

def clean_text(text):
    """Cleans extracted text by removing unnecessary characters and whitespace."""
    text = re.sub(r'\s+', ' ', text)  # Remove excessive whitespace
    text = re.sub(r'[^\w\s.,!?-]', '', text)  # Remove special characters (except punctuation)
    return text.strip()

def summarize_text(text, max_tokens=5000):  # Keep within safe limits
    """Trims long text and sends to Groq API for summarization."""
    trimmed_text = text[:max_tokens]  # Reduce text size

    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": "llama-3.1-8b-instant",  # Adjust model if needed
        "messages": [{"role": "system", "content": "Summarize the following document concisely."},
                     {"role": "user", "content": trimmed_text}]
    }

    response = requests.post(GROQ_API_URL, headers=headers, json=payload)

    # Debugging: Print response to check token size
    print("API Response:", response.json())

    return response.json().get("choices", [{}])[0].get("message", {}).get("content", "Summarization failed.")



# Example usage
pdf_path = "Notes with AI (1).pdf"  # Change this to your PDF file path
text = extract_text_from_pdf(pdf_path)
summary = summarize_text(text)
print("Summary:", summary)


API Response: {'id': 'chatcmpl-a19def75-40eb-43f6-b06c-d617f5b300c0', 'object': 'chat.completion', 'created': 1739084248, 'model': 'llama-3.1-8b-instant', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Here is a concise summary of the document:\n\n**Notes with AI Enhance, Quiz, and Excel with AI-Driven Notes**\n\n**Motivation:** The project aims to address challenges faced by students in managing, organizing, and retaining vast amounts of information in the digital learning environment.\n\n**Objective:** To revolutionize the learning process by leveraging artificial intelligence to automate and enhance key study tasks, improving student efficiency and learning outcomes.\n\n**Key Functionalities:**\n\n1. Transcript Fetching and Lecture Recording\n2. Summarization and Note-Making using AI\n3. Quiz Generation using AI models like BERT and SpaCy\n4. Flashcard Creation and Interactive QA system\n5. Save Notes and Organization\n\n**Methodology:** The project uses a com